# 18.4 Pitfalls of nonlinear programming

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

While AMPL gives you the power to formulate diverse nonlinear optimization models,
no solver can guarantee an acceptable solution every time you type solve. The algorithms
used by solvers are susceptible to a variety of difficulties inherent in the complexities
of nonlinear functions. This is in unhappy contrast to the linear case, where a welldesigned
solver can be relied upon to solve almost any linear program.

This section offers a brief introduction to the pitfalls of nonlinear programming. We
focus on two common kinds of difficulties, function range violations and multiple local
optima, and then mention several other traps more briefly.

For illustration we begin with the nonlinear transportation model shown in Figure
**18-4.** It is identical to our earlier transportation example ([Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a)) except that the
terms cost[i,j] * Trans[i,j] are replaced by nonlinear terms in the objective:

```ampl
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
       rate[i,j] * Trans[i,j] / (1 - Trans[i,j]/limit[i,j]);
```

Each term is a convex increasing function of Trans[i,j] like that depicted in [Figure 18-2e](./18_1_sources_of_nonlinearity.ipynb#fig-18-2e);
it is approximately linear with slope rate[i,j] at relatively small values of
Trans[i,j], but goes to infinity as Trans[i,j] approaches limit[i,j]. Associated
data values, also similar to those used for the linear transportation example in
[Chapter 3](../03/03.md), are given in [Figure 18-5](./18_4_pitfalls_of_nonlinear_programming.ipynb#fig-18-5).

**Function range violations**

An attempt to solve using the model and data as given proves unsuccessful:

```ampl
ampl: model nltrans.mod;
ampl: data nltrans.dat;
ampl: solve;
MINOS 5.5 Error evaluating objective Total_Cost
can't compute 8000/0
MINOS 5.5: solution aborted.

8 iterations, objective 0
```

The solver's message is cryptic, but strongly suggests a division by zero while evaluating
the objective. That could only happen if the expression

```ampl
1 - Trans[i,j]/limit[i,j]
```

```ampl
set ORIG;                       # origins
set DEST;                       # destinations
param supply {ORIG} >= 0;       # amounts available at origins
param demand {DEST} >= 0;       # amounts required at destinations
              check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];
param rate {ORIG,DEST} >= 0;    # base shipment costs per unit
param limit {ORIG,DEST} > 0;    # limit on units shipped
var Trans {i in ORIG, j in DEST} >= 0; # units to ship
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
              rate[i,j] * Trans[i,j] / (1 - Trans[i,j]/limit[i,j]);
subject to Supply {i in ORIG}:
       sum {j in DEST} Trans[i,j] = supply[i];
subject to Demand {j in DEST}:
       sum {i in ORIG} Trans[i,j] = demand[j];
```

**[Figure 18-4](./18_4_pitfalls_of_nonlinear_programming.ipynb#fig-18-4):** Nonlinear transportation model (nltrans.mod).

```ampl
param: ORIG:    supply :=
              GARY     1400    CLEV    2600  PITT  2900 ;
param: DEST:    demand :=
              FRA       900    DET     1200  LAN    600
              WIN       400    STL     1700  FRE   1100
              LAF      1000 ;
param rate : FRA   DET   LAN   WIN   STL  FRE   LAF :=
       GARY   39   14    11    14    16    82     8
       CLEV   27    9    12     9    26    95    17
       PITT   24   14    17    13    28    99     20 ;
param limit :     FRA DET LAN WIN     STL   FRE LAF :=
        GARY      500 1000 1000 1000  800   500 1000
        CLEV      500 800 800 800     500   500 1000
        PITT      800 600 600 600     500   500 900 ;
```

**[Figure 18-5](./18_4_pitfalls_of_nonlinear_programming.ipynb#fig-18-5):** Data for nonlinear transportation model (nltrans.dat).

is zero at some point. If we use display to print the pairs where Trans[i,j] equals
limit[i,j]:

```ampl
ampl: display {i in ORIG, j in DEST: Trans[i,j] = limit[i,j]};
set {i in ORIG, j in DEST: Trans[i,j] == limit[i,j]}
              := (GARY,LAF) (PITT,LAN);
ampl: display Trans['GARY','LAF'], limit['GARY','LAF'];
Trans['GARY','LAF'] = 1000
limit['GARY','LAF'] = 1000
```

we can see the problem. The solver has allowed Trans[GARY,LAF] to have the value 1000,
which equals limit[GARY,LAF]. As a result, the objective function term

```ampl
rate[GARY,LAF] * Trans[GARY,LAF]
       / (1 - Trans[GARY,LAF]/limit[GARY,LAF])
```

evaluates to 8000/0. Since the solver is unable to evaluate the objective function, it gives
up without finding an optimal solution.

Because the behavior of a nonlinear optimization algorithm can be sensitive to the
choice of starting guess, we might hope that the solver will have greater success from a
different start. To ensure that the comparison is meaningful, we first set

```ampl
ampl: option send_statuses 0;
```

so that status information about variables that was returned by the previous solve will not
be sent back to help determine a starting point for the next solve. Then AMPL's let
command may be used to suggest, for example, a new initial value for each
Trans[i,j] that is half of limit[i,j]:

```ampl
ampl: let {i in ORIG, j in DEST} Trans[i,j] := limit[i,j]/2;
ampl: solve;
MINOS 5.5: the current point cannot be improved.

32 iterations, objective -7.385903389e+18
```

This time the solver runs to completion, but there is still something wrong. The objective
is less than − 10 18 , or − ∞ for all practical purposes, and the solution is described as "cannot
be improved" rather than optimal.

Examining the values of Trans[i,j]/limit[i,j] in the solution that the solver
has returned gives a clue to the difficulty:

```ampl
ampl: display {i in ORIG, j in DEST} Trans[i,j]/limit[i,j];
Trans[i,j]/limit[i,j] [*,*] (tr)
:      CLEV        GARY          PITT :=
DET   -6.125e-14   0             2
FRA    0           1.5           0.1875
FRE    0.7         1             0.5
LAF    0.4         0.15          0.5
LAN    0.375       7.03288e-15   0.5
STL    2.9         0             0.5
WIN    0.125       0             0.5
;
```

These ratios show that the shipments for several pairs, such as Trans[CLEV,STL],
significantly exceed their limits. More seriously, Trans[GARY,FRE] seems to be right
at limit[GARY,FRE], since their ratio is given as 1. If we display them to full precision,
however, we see:

```ampl
ampl: option display_precision 0;
ampl: display Trans['GARY','FRE'], limit['GARY','FRE'];
Trans['GARY','FRE'] = 500.0000000000028
limit['GARY','FRE'] = 500
```

**[Figure 18-6](./18_4_pitfalls_of_nonlinear_programming.ipynb#fig-18-6):** Singularity in cost function y = x /( 1 − x / c).

The variable is just slightly larger than the limit, so the cost term has a huge negative
value. If we graph the entire cost function, as in [Figure 18-6](./18_4_pitfalls_of_nonlinear_programming.ipynb#fig-18-6), we see that indeed the cost
function goes off to − ∞ to the right of the singularity at limit[GARY,FRE].

The source of error in both runs above is our assumption that, since the objective goes
to + ∞ as Trans[i,j] approaches limit[i,j] from below, the solver will keep
Trans[i,j] between 0 and limit[i,j]. At least for this solver, we must enforce
such an assumption by giving each Trans[i,j] an explicit upper bound that is slightly
less than limit[i,j], but close enough not to affect the eventual optimal solution:

```ampl
var Trans {i in ORIG, j in DEST} >= 0, <= .9999 * limit[i,j];
```

With this modification, the solver readily finds an optimum:

```ampl
ampl: option display_precision 6;
ampl: model nltransb.mod; data nltrans.dat; solve;
MINOS 5.5: optimal solution found.

81 iterations, objective 1212117
ampl: display Trans;
Trans [*,*] (tr)
:       CLEV       GARY      PITT      :=
DET   586.372    187.385    426.242
FRA   294.993     81.2205   523.787
FRE   365.5      369.722    364.778
LAF   490.537      0        509.463
LAN   294.148      0        305.852
STL   469.691    761.672    468.637
WIN    98.7595     0        301.241
;
```

These values of the variables are well away from any singularity, with
Trans[i,j]/limit[i,j] being less than 0.96 in every case. (If you change the
starting guess to be limit[i,j]/2 as before, you should find that the solution is the
same but the solver needs only about half as many iterations to find it.)
The immediate lesson here is that nonlinear functions can behave quite badly outside
the intended range of the variables. The incomplete graph in [Figure 18-2e](./18_1_sources_of_nonlinearity.ipynb#fig-18-2e) made this cost
function look misleadingly well-behaved, whereas [Figure 18-6](./18_4_pitfalls_of_nonlinear_programming.ipynb#fig-18-6) shows the need for a
bound to keep the variable away from the singularity.

A more general lesson is that difficulties posed by a nonlinear function may lead the
solver to fail in a variety of ways. When developing a nonlinear model, you need to be
alert to bizarre results from the solver, and you may have to do some detective work to
trace the problem back to a flaw in the model.

**Multiple local optima**

To illustrate a different kind of difficulty in nonlinear optimization, we consider a
slightly modified objective function that has the following formula:

```ampl
minimize Total_Cost:
       sum {i in ORIG, j in DEST}
       rate[i,j] * Trans[i,j]ˆ0.8 / (1 - Trans[i,j]/limit[i,j]);
```

By raising the amount shipped to the power 0.8, we cause the cost function to be concave
at lower shipment amounts and convex at higher amounts, in the manner of [Figure 18-2f](./18_1_sources_of_nonlinearity.ipynb#fig-18-2f).
Attempting to solve this new model, we again initially run into technical difficulties:

```ampl
ampl: model nltransc.mod; data nltrans.dat; solve;
MINOS 5.5: Error evaluating objective Total_Cost:
                            can't evaluate pow'(0,0.8)
MINOS 5.5: solution aborted.

8 iterations, objective 0
```

This time our suspicion naturally centers upon Trans[i,j]ˆ0.8, the only expression
that we have changed in the model. A further clue is provided by the error message's reference
to pow'(0,0.8), which denotes the derivative of the exponential (power) function
at zero. When Trans[i,j] is zero, this function has a well-defined value, but its
derivative with respect to the variable — the slope of the graph in [Figure 18-2f](./18_1_sources_of_nonlinearity.ipynb#fig-18-2f) — is infinite.
As a result, the partial derivative of the total cost with respect to any variable at zero
cannot be returned to the solver; since the solver requires all the partial derivatives for its
optimization algorithm, it gives up.

This is another variation on the range violation problem, and again it can be remedied
by imposing some bounds to keep the solution away from troublesome points. In this
case, we move the lower bound from zero to a very small positive number:

```ampl
var Trans {i in ORIG, j in DEST}
       >= 1e-10, <= .9999 * limit[i,j], := 0;
```

We might also move the starting guess away from zero, but in this example the solver
takes care of that automatically, since the initial values only suggest a starting point.

With the bounds adjusted, the solver runs normally and reports a solution:

```ampl
ampl: model nltransd.mod; data nltrans.dat; solve;
MINOS 5.5: optimal solution found.

65 iterations, objective 427568.1225
ampl: display Trans;
Trans [*,*] (tr)
:       CLEV         GARY       PITT       :=
DET     689.091   1e-10         510.909
FRA   1e-10          199.005    700.995
FRE     385.326     326.135     388.54
LAF     885.965     114.035   1e-10
LAN     169.662   1e-10         430.338
STL     469.956     760.826     469.218
WIN   1e-10       1e-10         400
;
```

We can regard each 1e-10 as a zero, since such a small value is negligible in comparison
with the rest of the solution.

Next we again try a starting guess at limit[i,j]/2, in the hope of speeding things
up. This is the result:

```ampl
ampl: let {i in ORIG, j in DEST} Trans[i,j] := limit[i,j]/2;
ampl: solve;
MINOS 5.5: optimal solution found.

40 iterations, objective 355438.2006
ampl:   display Trans;
Trans   [*,*] (tr)
:        CLEV       GARY        PITT :=
DET     540.601     265.509    393.89
FRA     328.599   1e-10        571.401
FRE     364.639     371.628    363.732
LAF     491.262   1e-10        508.738
LAN     301.741   1e-10        298.259
STL     469.108     762.863    468.029
WIN     104.049   1e-10        295.951
;
```

Not only is the solution completely different, but the optimal value is 17% lower! The
first solution could not truly have minimized the objective over all solutions that are feasible
in the constraints.

Actually both solutions can be considered correct, in the sense that each is locally
optimal. That is, each solution is less costly than any other nearby solutions. All of the
classical methods of nonlinear optimization, which are the methods considered in this
chapter, are designed to seek a local optimum. Started from one specified initial guess,
these methods are not guaranteed to find a solution that is globally optimal, in the sense
of giving the best objective value among all solutions that satisfy the constraints. In general,
finding a global optimum is much harder than finding a local one.
Fortunately, there are many cases in which a local optimum is entirely satisfactory.
When the objective and constraints satisfy certain properties, any local optimum is also
global; the model considered at the beginning of this section is one example, where the
convexity of the objective, together with the linearity of the constraints, guarantees that
the solver will find a global optimum. (Linear programming is an even more special case
with this property; that's why in previous chapters we never encountered local optima
that were not global.)

Even when there is more than one local optimum, a knowledge of the situation being
modeled may help you to identify the global one. Perhaps you can choose an initial solution
near to the global optimum, or you can add some constraints that rule out regions
known to contain local optima.

Finally, you may be content to find a very good local optimum, even if you don't
have a guarantee that it is global. One straightforward approach is to try a series of starting
points systematically, and take the best among the solutions. As a simple illustration,
suppose that we declare the variables in our example as follows:

```ampl
param alpha >=0, <= 1;
var Trans {i in ORIG, j in DEST}
       >= 1e-10, <= .9999 * limit[i,j], := alpha * limit[i,j];
```

For each choice of alpha we get a different starting guess, and potentially a different
solution. Here are some resulting objective values for alpha ranging from 0 to 1:

```ampl
alpha   Total_Cost
 0.0      427568.1
 0.1      366791.2
 0.2      366791.2
 0.3      366791.2
 0.4      366791.2
 0.5      355438.2
 0.6      356531.5
 0.7      376043.3
 0.8      367014.4
 0.9      402795.3
 1.0      365827.2
```

The solution that we previously found for an alpha of 0.5 is still the best, but in light of
these results we are now more inclined to believe that it is a very good solution. We
might also observe that, although the reported objective value varies somewhat erratically
with the choice of starting point — a feature of nonlinear programs generally — the
second-best value of Total_Cost was found by setting alpha to 0.6. This suggests
that a closer search of alpha values around 0.5 might be worthwhile.

Some of the more sophisticated methods for global optimization attempt to search
through starting points in this way, but with a more elaborate and systematic procedure
for deciding which starting points to try next. Others treat global optimization as more of
a combinatorial problem, and apply solution methods motivated by those for integer programming
([Chapter 20](../20/20.md)). Global optimization methods are still at a relatively early stage
of development, and are likely to improve as experience accumulates, new ideas are tried,
and available computing power further increases.
**Other pitfalls**

Many other factors can influence the efficiency and success of a nonlinear solver,
including the way that the model is formulated and the choice of units (or scaling) for the
variables. As a rule, nonlinearities are more easily handled when they appear in the
objective function rather than in the constraints. AMPL's option to substitute variables
automatically, described earlier in this chapter, may help in this regard. Another rule of
thumb is that the values of the variables should differ by at most a few orders of magnitude;
solvers can be misled when some variables are, say, in millions and others are in
thousandths. Some solvers automatically scale a problem to try to avoid such a situation,
but you can help them considerably by judiciously picking the units in which the variables
are expressed.

Nonlinear solvers also have many modes of failure besides the ones we have discussed.
Some methods of nonlinear optimization can get stuck at "stationary" points
that are not optimal in any sense, can identify a maximum when a minimum is desired (or
vice-versa), and can falsely give an indication that there is no feasible solution to the constraints.
In these cases your only recourse may be to try a different starting guess; it can
sometimes help to specify a start that is feasible for many of the nonlinear constraints.
You may also improve the solver's chances of success by placing realistic bounds on the
variables. If you know, for instance, that an optimal value of 80 is plausible for some
variables, but a value of 800 is not, you may want to give them a bound of 400. (Once an
indicated optimum is at hand, you should be sure to `check` whether these "safety"
bounds have been reached by any of the variables; if so, the bounds should be relaxed and
the problem re-solved.)

The intent of this section has been to illustrate that extra caution is advisable in working
with nonlinear models. If you encounter a difficulty that cannot be resolved by any of
the simple devices described here, you may need to consult a textbook in nonlinear programming,
the documentation for the particular solver that you are using, or a numerical
analyst versed in nonlinear optimization techniques.

**Bibliography**

Roger Fletcher, Practical Methods of Optimization. John Wiley & Sons (New York, NY, 1987). A
concise survey of theory and methods.

Philip E. Gill, Walter Murray and Margaret H. Wright, Practical Optimization. Academic Press
(New York, NY, 1981). Theory, algorithms and practical advice.

Jorge Nocedal and Stephen J. Wright, Numerical Optimization. Springer Verlag (Heidelberg,
1999). A text on methods for optimization of smooth functions.
Richard P. O'Neill, Mark Williard, Bert Wilkins and Ralph Pike, "A Mathematical Programming
Model for Allocation of Natural Gas." Operations Research 27, 5 (1979) pp. 857-873. A source
for the nonlinear relationships in natural gas pipeline networks described in [Section 18.1](./18_1_sources_of_nonlinearity.ipynb).